In [26]:
# Instalación y carga del paquete wooldridge
if(!require(wooldridge)){
  install.packages("wooldridge")
  library(wooldridge)
}

# Fijar semilla para reproducibilidad y preparar datos
set.seed(1)
data(ceosal1)
ceosal1 <- na.omit(ceosal1)


In [27]:
# 1.2 Dummy de ventas sobre el promedio
ceosal1$sales_mayor_promedio <- ifelse(ceosal1$sales > mean(ceosal1$sales), 1, 0)

# 1.3 Interacción Dummy x Dummy
ceosal1$sales_mayor_promedio_indus <- ceosal1$sales_mayor_promedio * ceosal1$indus

# 1.4 Interacción Dummy x Continua
ceosal1$sales_mayor_promedio_roe <- ceosal1$sales_mayor_promedio * ceosal1$roe

# 1.5 Dummy de sector agrupado
ceosal1$indus_o_finance <- ceosal1$indus + ceosal1$finance

1.1 Descripción de las variables
La base ceosal1 se emplea habitualmente en economía laboral y gobierno corporativo para estudiar si la compensación de los ejecutivos está determinada por el tamaño de la firma o por su rendimiento financiero.

    Dependiente (Continua): salary (Salario del CEO en miles de USD).

    Explicativas Continuas: sales (Ventas en millones de USD, proxy de tamaño), roe (Retorno sobre el capital en %), ros (Retorno sobre la acción en %).

    Explicativas Dummies: indus, finance, consprod, utility (Toman el valor 1 si la empresa pertenece a esa industria, 0 si no).


Procedimiento y resultado:

    La variable sales_mayor_promedio_indus (1.3) indica el efecto conjunto de pertenecer a la industria y, simultáneamente, tener ventas sobre el promedio.

    La interacción sales_mayor_promedio_roe (1.4) evalúa si el impacto del ROE sobre el salario cambia estructuralmente cuando la empresa es grande (ventas sobre el promedio) respecto a cuando es pequeña.

In [28]:
# Estimación del modelo base
modelo_base <- lm(salary ~ sales + roe + ros, data=ceosal1)
summary(modelo_base)


Call:
lm(formula = salary ~ sales + roe + ros, data = ceosal1)

Residuals:
    Min      1Q  Median      3Q     Max 
-1495.0  -486.5  -242.8    98.9 13516.4 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) 864.117516 228.399737   3.783 0.000203 ***
sales         0.015482   0.008954   1.729 0.085291 .  
roe          22.003312  11.516554   1.911 0.057454 .  
ros          -1.105191   1.450240  -0.762 0.446892    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1360 on 205 degrees of freedom
Multiple R-squared:  0.03191,	Adjusted R-squared:  0.01775 
F-statistic: 2.253 on 3 and 205 DF,  p-value: 0.08337


Procedimiento y resultado:
Al correr este modelo, típicamente se verá que sales y roe tienen p-values pequeños (son significativos). El test F global te dirá que, en conjunto, el tamaño de la firma y su rendimiento explican una parte real de la variación de los salarios. Sin embargo, en la práctica, suele verse que el tamaño de la empresa (sales) tiene un peso mucho mayor en el sueldo que el rendimiento (roe), lo cual es un hallazgo clásico sobre los problemas de agencia corporativa.
Al ejecutar este código, el summary te mostrará los coeficientes (Estimates) y sus p-values (Pr(>|t|)). Las variables sales y roe suelen tener p-values < 0.05, indicando que afectan significativamente al salario. Al final del resumen, el F-statistic evalúa la significancia global; si su p-value es < 0.05, el modelo sí tiene significancia global, confirmando que las variables en conjunto explican las variaciones del salario.

En este caso como el p-value es 0.08337, no hay significancia global, por lo tanto solo explican una parte de la variación de los salarios.


### 1. La Significancia Individual (`Pr(>|t|)`)
Esta es la parte más importante para responder la regla del 0.05
*   **sales (0.085):** Es mayor que 0.05.
*   **roe (0.057):** Es mayor que 0.05.
*   **ros (0.446):** Es muchísimo mayor que 0.05.

**Conclusión estadística:** Al 5% de significancia, **ninguna** de las variables explicativas es estadísticamente significativa de forma individual. R les puso un punto (`.`) en lugar de un asterisco (`*`), lo que indica que `sales` y `roe` son significativas solo si fuéramos más flexibles y usáramos un 10% de margen de error. Al estricto 5% que nos pide el modelo , no hay evidencia suficiente de que afecten el salario.

### 2. La Significancia Global (`F-statistic`)
Mira la última línea del reporte: `p-value: 0.08337`.
*   Como **0.08337 es mayor que 0.05**, no podemos rechazar la hipótesis nula conjunta.
*   **Conclusión:** El modelo **NO tiene significancia global al 5%**[cite: 2]. Esto significa que, en conjunto, las ventas, el ROE y el ROS no son suficientes para predecir el salario de los CEOs de manera confiable bajo este nivel de exigencia.

### 3. El Ajuste del Modelo (`Multiple R-squared`)
*   La $R^2$ es de **0.03191**.
*   **Conclusión económica:** Este modelo base es bastante pobre. Las ventas y los retornos (ROE/ROS) apenas logran explicar el **3.19%** de las diferencias salariales entre los CEOs de la muestra. El ~97% del salario de un CEO se explica por factores que no están en este modelo (como el sector de la empresa, su experiencia, o bonos atípicos).

### 4. Los Coeficientes (`Estimate`)
Aunque estadísticamente no pasaron la prueba del 5%, la interpretación económica (si lo diéramos por válido) sería esta:
*   **`(Intercept)` (864.11):** El salario base autónomo sería de unos $864,117 dólares.
*   **`sales` (0.015):** Por cada millón de dólares extra en ventas, el salario del CEO sube apenas ~$15.48 dólares.
*   **`roe` (22.00):** Por cada 1% adicional en el retorno sobre el capital, el salario sube ~$22,003 dólares.

---



In [29]:
# Modelo incluyendo la dummy y la interacción con roe
modelo_ext <- lm(salary ~ sales + roe + ros + sales_mayor_promedio + sales_mayor_promedio_roe, data=ceosal1)
summary(modelo_ext)


Call:
lm(formula = salary ~ sales + roe + ros + sales_mayor_promedio + 
    sales_mayor_promedio_roe, data = ceosal1)

Residuals:
    Min      1Q  Median      3Q     Max 
 -976.0  -469.1  -263.4    97.2 13566.3 

Coefficients:
                           Estimate Std. Error t value Pr(>|t|)   
(Intercept)              783.298711 258.064395   3.035  0.00272 **
sales                      0.006657   0.011197   0.594  0.55285   
roe                       24.472338  13.088222   1.870  0.06295 . 
ros                       -0.986904   1.457818  -0.677  0.49919   
sales_mayor_promedio     513.083213 510.134319   1.006  0.31572   
sales_mayor_promedio_roe  -9.498301  25.587585  -0.371  0.71087   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1361 on 203 degrees of freedom
Multiple R-squared:  0.04059,	Adjusted R-squared:  0.01696 
F-statistic: 1.718 on 5 and 203 DF,  p-value: 0.132


Procedimiento y resultado:
Al ver el coeficiente de sales_mayor_promedio aislado, se puede ver el salto en el salario base solo por estar en una empresa grande. Al ver el coeficiente de la interacción (sales_mayor_promedio_roe), ves si el CEO de una empresa grande recibe un extra de dólares por cada punto de rentabilidad respecto al CEO de una empresa pequeña.
Aquí evaluamos cómo cambian los salarios para las firmas con altas ventas. El coeficiente de sales_mayor_promedio indica la diferencia en el salario base (intercepto) entre un CEO de una firma grande vs. una pequeña. El coeficiente de sales_mayor_promedio_roe indica el cambio en la pendiente; es decir, cuánto premio salarial adicional recibe el CEO por cada punto extra de ROE, solo por estar en una empresa grande.



### Respuesta Pregunta 2.2: Interpretación del Modelo Extendido

**1. Impacto económico de las variables agregadas en el salario:**
Al incluir la variable *dummy* (`sales_mayor_promedio`) y su interacción con el rendimiento (`sales_mayor_promedio_roe`), el modelo intenta capturar si el esquema de compensación cambia para los CEOs de empresas con ventas sobre el promedio.

*   **Efecto en el salario base (`sales_mayor_promedio`):** El coeficiente estimado es de **513.08**. Esto indica que, manteniendo todo lo demás constante, los CEOs que dirigen empresas con ventas superiores al promedio tienen un salario base de 513,083 dólares más alto que los CEOs de empresas con ventas bajo el promedio.
*   **Efecto en la pendiente del rendimiento (`sales_mayor_promedio_roe`):** El coeficiente estimado es de **-9.49**. Esto señala que, en las empresas con ventas sobre el promedio, la recompensa salarial por cada punto porcentual adicional de ROE es menor que en las empresas pequeñas. Mientras un CEO de empresa pequeña gana 24,472 dólares adicionales por cada punto de ROE, el CEO de una empresa grande gana solo 14,974 dólares (obtenido de 24.472 - 9.498).

**2. Significancia estadística individual:**
A pesar de las grandes magnitudes económicas descritas anteriormente, **ninguna de las variables agregadas es estadísticamente significativa al 5%**.
*   El *p-value* de la variable *dummy* es 0.315 (mayor a 0.05).
*   El *p-value* de la interacción es 0.710 (mayor a 0.05).
Esto implica que no existe evidencia estadística suficiente para afirmar que las empresas grandes pagan un bono base diferente o premian el ROE de manera distinta a las empresas pequeñas. De hecho, en este nuevo modelo, ninguna variable explicativa resulta significativa al 5% (el *p-value* más bajo lo tiene `roe` con 0.062).

**3. Significancia global y bondad de ajuste:**
*   El modelo en su conjunto sigue **sin ser globalmente significativo al 5%**, ya que el *p-value* del estadístico F es **0.132** (mayor a 0.05).
*   Además, aunque el $R^2$ múltiple subió marginalmente a 4.05%, el **$R^2$ ajustado disminuyó** de 1.77% (en el modelo base) a **1.69%**. Esta caída en el $R^2$ ajustado nos indica que penalizó la inclusión de estas dos nuevas variables, confirmando que no aportan poder explicativo real al modelo y solo consumen grados de libertad.

***



In [30]:
# Extraemos los R-cuadrados
R2_r <- summary(modelo_base)$r.squared
R2_ur <- summary(modelo_ext)$r.squared

# n = 209, k = 5, q = 2 (se agregaron 2 variables)
F_critico <- qf(0.95, 2, 203)
F_est <- ((R2_ur - R2_r) / 2) / ((1 - R2_ur) / 203)

cat("Estadístico F:", F_est, "| F Crítico:", F_critico, "\n")
if (F_est > F_critico) {
  print("Se rechaza H0: Las variables agregadas son conjuntamente significativas.")
} else {
  print("No se rechaza H0.")
}

Estadístico F: 0.9177601 | F Crítico: 3.040379 
[1] "No se rechaza H0."


Procedimiento y resultado:
Comparamos el poder explicativo (R2) de ambos modelos. Si el estadístico F calculado es mayor al F crítico de la tabla de distribución (o regla del 5%), significa que añadir la dummy y la interacción mejoró genuinamente el modelo.
El Test F penaliza la pérdida de grados de libertad. Si el R2 del modelo 2.2 subió lo suficiente en comparación al 2.1, el estadístico F superará al F crítico. Si se rechaza la hipótesis nula (H0​), se concluye que el tamaño de las ventas y su interacción con el ROE son determinantes clave que no pueden omitirse al estudiar el salario
Como el estadístico F calculado es mayor al F crítico de la tabla, esto significa que al añadir la dummy la interacción no mejoró significativamente al modelo, no se rechaza la hipótesis nula.


***

### Respuesta Pregunta 2.3: Interpretación del Test de Hipótesis Conjunta (Test F)

**1. Planteamiento de las Hipótesis:**
En este inciso se evalúa si las dos variables agregadas en el modelo anterior (`sales_mayor_promedio` y `sales_mayor_promedio_roe`) aportan información valiosa al modelo cuando se consideran al mismo tiempo.
*   **Hipótesis Nula ($H_0$):** Ambos coeficientes son iguales a cero ($\beta_4 = 0$ y $\beta_5 = 0$). Es decir, las variables no son conjuntamente significativas.
*   **Hipótesis Alternativa ($H_1$):** Al menos uno de los coeficientes es distinto de cero.

**2. Regla de Decisión y Resultado Estadístico:**
Para tomar una decisión con un nivel de significancia del 5%[cite: 2], comparamos el valor del estadístico F calculado con el valor crítico extraído de la distribución F:
*   **Estadístico F calculado:** 0.9177
*   **F Crítico (de tabla):** 3.0403

Dado que el estadístico F calculado es estrictamente menor que el F crítico ($0.9177 < 3.0403$), el valor cae en la zona de no rechazo. Por lo tanto, la decisión estadística es que **No se rechaza $H_0$**.

**3. Conclusión Económica y Econométrica:**
La conclusión es que **las variables agregadas no son conjuntamente significativas** para explicar el salario de los CEOs.

Esto confirma matemáticamente lo que ya sugería el modelo anterior: diferenciar a las empresas en dos grupos (las que venden por sobre el promedio y las que no) e interactuar esta condición con el ROE no mejora la capacidad predictiva del modelo. La inclusión de estas variables no aporta nuevo poder explicativo real y, desde una perspectiva econométrica, el modelo base (solo con `sales`, `roe` y `ros`) es preferible ya que es más parsimonioso.

***


In [31]:
# Creación de interacciones para todas las variables base
ceosal1$sales_D <- ceosal1$sales * ceosal1$sales_mayor_promedio
ceosal1$roe_D <- ceosal1$roe * ceosal1$sales_mayor_promedio
ceosal1$ros_D <- ceosal1$ros * ceosal1$sales_mayor_promedio

# Modelo sin restringir con todas las interacciones
modelo_interacciones <- lm(salary ~ sales + roe + ros + sales_mayor_promedio + sales_D + roe_D + ros_D, data=ceosal1)

R2_ur2 <- summary(modelo_interacciones)$r.squared
# q = 4 restricciones (1 dummy de intercepto + 3 interacciones), gl = 209 - 7 - 1 = 201
F_critico2 <- qf(0.95, 4, 201)
F_est2 <- ((R2_ur2 - R2_r) / 4) / ((1 - R2_ur2) / 201)

cat("Estadístico F:", F_est2, "| F Crítico:", F_critico2, "\n")
if (F_est2 > F_critico2) {
  print("Se rechaza H0: Hay un quiebre estructural entre firmas de altas y bajas ventas.")
} else {
  print("No se rechaza H0: El comportamiento de los salarios es el mismo en ambos grupos.")
}

Estadístico F: 1.517033 | F Crítico: 2.416574 
[1] "No se rechaza H0: El comportamiento de los salarios es el mismo en ambos grupos."


Procedimiento y resultado:
Este enfoque evalúa un cambio estructural completo. Al interactuar la dummy de grupo con todos los regresores, testeamos si las funciones de compensación salarial operan bajo reglas distintas dependiendo de si la empresa vende por encima o por debajo del promedio.
Al interactuar la dummy de tamaño con todas las variables, estás partiendo tu regresión en dos implícitamente. Si el Test F rechaza H0​, significa que existe un quiebre estructural. La fórmula para calcular el salario de un CEO en una empresa grande no tiene nada que ver con la fórmula que usa una empresa que vende por debajo del promedio.



### Respuesta Pregunta 2.4: Interpretación del Test de Hipótesis para Grupos (Test de Quiebre Estructural)

**1. Planteamiento de las Hipótesis:**
En este inciso se busca evaluar si existe un cambio estructural en la forma en que se determina el salario dependiendo del tamaño de la empresa (Grupo 1: ventas mayores al promedio vs. Grupo 2: ventas menores al promedio)[cite: 2].
*   **Hipótesis Nula ($H_0$):** No existen diferencias estructurales entre los grupos. Es decir, los retornos marginales de las variables (`sales`, `roe`, `ros`) y el salario base (intercepto) son estadísticamente iguales para ambos grupos de empresas.
*   **Hipótesis Alternativa ($H_1$):** Existe un quiebre estructural. Al menos uno de los parámetros que determinan la compensación salarial es diferente entre las empresas de altas ventas y las de bajas ventas.

**2. Regla de Decisión y Resultado Estadístico:**
Utilizando un nivel de significancia del 5%[cite: 2], contrastamos el estadístico F obtenido a partir de la comparación del modelo restringido (base) y el modelo irrestricto (con todas las interacciones) contra el valor crítico de la distribución F:
*   **Estadístico F calculado:** 1.5170
*   **F Crítico (de tabla):** 2.4165

Puesto que el estadístico F calculado es estrictamente menor que el F crítico ($1.5170 < 2.4165$), el valor se encuentra en la región de aceptación. Por lo tanto, la decisión estadística es **No rechazar $H_0$**.

**3. Conclusión Económica y Econométrica:**
Empíricamente, concluimos que **el comportamiento de los salarios es estadísticamente el mismo en ambos grupos**. Esto significa que la estructura de compensación que usan las empresas para pagar a sus CEOs (cómo valoran y premian el nivel de ventas, el ROE y el ROS) no cambia de forma significativa por el hecho de que la empresa venda por encima o por debajo del promedio.

Desde el punto de vista econométrico, estimar modelos separados por tamaño de ventas (o usar un modelo lleno de interacciones) no aporta una mejora significativa frente al modelo base unificado.

***



In [32]:
# Submuestras por sector
ceosal1_indus <- subset(ceosal1, indus==1)
ceosal1_finance <- subset(ceosal1, finance==1)
ceosal1_consprod <- subset(ceosal1, consprod==1)
ceosal1_utility <- subset(ceosal1, utility==1)

# Modelos por subgrupo
m_indus <- lm(salary ~ sales + roe + ros, data=ceosal1_indus)
m_finance <- lm(salary ~ sales + roe + ros, data=ceosal1_finance)
m_consprod <- lm(salary ~ sales + roe + ros, data=ceosal1_consprod)
m_utility <- lm(salary ~ sales + roe + ros, data=ceosal1_utility)

# Extraer las Sumas de Residuos al Cuadrado (SSR)
SSR_r <- sum(residuals(modelo_base)^2)
SSR_indus <- sum(residuals(m_indus)^2)
SSR_finance <- sum(residuals(m_finance)^2)
SSR_consprod <- sum(residuals(m_consprod)^2)
SSR_utility <- sum(residuals(m_utility)^2)

# SSR irrestricto es la suma de los subgrupos
SSR_ur <- SSR_indus + SSR_finance + SSR_consprod + SSR_utility

# Grados de libertad: G=4 grupos, k=3 variables
G <- 4
k <- 3
n <- nrow(ceosal1)
gl1 <- (G - 1) * (k + 1)
gl2 <- n - G * (k + 1)

# Cálculo de F
F_critico_chow <- qf(0.95, gl1, gl2)
F_est_chow <- ((SSR_r - SSR_ur) / gl1) / (SSR_ur / gl2)

cat("Estadístico F Chow:", F_est_chow, "| F Crítico:", F_critico_chow, "\n")
if (F_est_chow > F_critico_chow) {
  print("Se rechaza H0: Los modelos NO son iguales entre las distintas industrias.")
} else {
  print("No se rechaza H0.")
}

Estadístico F Chow: 1.398349 | F Crítico: 1.802614 
[1] "No se rechaza H0."


A diferencia de la pregunta 2.4, aquí usamos la técnica clásica de SSR (Suma de Cuadrados de los Residuos). El código calcula el error del modelo unificado (restringido) frente al error de hacer 4 modelos separados (irrestricto). Si modelar por separado reduce fuertemente el error (SSR), el test rechazará la hipótesis nula, probando que el sector económico cambia drásticamente la estructura de cómo se determinan los salarios de los CEO.
El Resultado: El Test F compara los "errores" (Residuos). Suma el error de obligar a todas las empresas a usar la misma línea de regresión (Modelo Restringido) y lo compara con la suma de los errores de dejar que cada sector tenga su propia línea (Modelo Irrestricto).

    Si rechazamos H0​, concluyimos empíricamente que el sector importa drásticamente. Por ejemplo, los CEOs financieros podrían estar sujetos a esquemas de bonos por ROE muy agresivos, mientras que los CEOs de servicios públicos (empresas más estables y reguladas) podrían tener salarios fijos donde el ROE casi no importa. El test confirma matemáticamente que no puedes agrupar a todos los sectores bajo la misma lupa.
    En este caso no se rechaza H0, por lo que el sector no importa drásticamente, por lo que el test nos confirmaría que podemos agrupar a todos los sectores bajo una misma mirada o visión, para así hacer un análisis más en conjunto que a través de la separación de las variables.


***

### Respuesta Pregunta 2.5: Interpretación del Test de Chow (Diferencias por Sector Económico)

**1. Planteamiento de las Hipótesis:**
En este ejercicio se aplica un Test de Chow clásico, basado en la comparación de la Suma de Cuadrados de los Residuos (SSR), para determinar si el sector económico al que pertenece la empresa genera un quiebre estructural en el modelo de compensación salarial.
*   **Hipótesis Nula ($H_0$):** Los parámetros de la regresión son estadísticamente iguales en todos los grupos. Es decir, la estructura salarial es la misma sin importar si el sector es industrial, financiero, de productos de consumo o de servicios públicos.
*   **Hipótesis Alternativa ($H_1$):** Al menos uno de los sectores tiene una estructura de compensación estadísticamente distinta a los demás.

**2. Regla de Decisión y Resultado Estadístico:**
Evaluando a un nivel de significancia del 5%, contrastamos el estadístico F de Chow calculado frente al valor crítico correspondiente:
*   **Estadístico F calculado:** 1.3983
*   **F Crítico (de tabla):** 1.8026

Dado que el estadístico F calculado es estrictamente menor que el F crítico ($1.3983 < 1.8026$), el valor recae en la zona de no rechazo. En consecuencia, la decisión estadística es **No rechazar $H_0$**.

**3. Conclusión Económica y Econométrica:**
La evidencia empírica indica que **los modelos son estadísticamente iguales entre las distintas industrias**. Desde un punto de vista económico, esto significa que el mercado valora y recompensa el desempeño y tamaño corporativo (a través de las ventas, el ROE y el ROS) de una manera transversal; a los CEOs se les paga bajo las mismas reglas base sin importar si dirigen un banco, una fábrica o una empresa de servicios públicos.

A nivel econométrico, esta conclusión es muy valiosa: demuestra que es correcto y eficiente utilizar el modelo base unificado (con toda la muestra junta) para explicar el salario. Separar la base de datos para correr regresiones individuales por industria no logra reducir significativamente el error agregado del modelo (SSR), justificando así la parsimonia del modelo general.

***
